# Clean PPG and ABP Windows and Extract BP Labels

Loads aligned 625-sample (5-second, 125 Hz) PPG and ABP windows from `data/processed_dataset.npz`, cleans both signals, labels blood pressure from the cleaned ABP, drops unusable rows from both datasets together, and saves `data/cleaned_dataset.npz`.

**Cleaning choices:** ABP gets a 4th-order Butterworth low-pass at 40 Hz through NeuroKit2. PPG uses the full NeuroKit2 `ppg_process` pipeline. Windows are excluded for non-finite values, flat signals, or more than 5% zero samples; remaining zeros are linearly interpolated. The cleaned ABP labels come from `extract_bp_labels()` in `data/bp_utils.py`: it detects prominent systolic peaks, takes each beat's diastolic value as the lowest ABP between consecutive peaks, and uses the median over complete beats (windows with fewer than two complete beats get NaN). Labels are then checked against the configured SBP/DBP ranges and `SBP > DBP`. No clipping or beat-to-beat variability rejection is applied.

**Label method change:** earlier versions averaged all local peaks and valleys, which counted small secondary bumps as heartbeats. On 3,000 training windows, the new method changed median SBP from 107.2 to 128.6 mmHg. It is improved but not clinically validated. A `cleaned_dataset.npz` saved before this change has the old labels: rerun this notebook, then retrain models such as the CNN. The saved `abp_valley_counts` array now holds the number of complete beats per window.

In [6]:
import sys
from collections import Counter
from pathlib import Path

import neurokit2 as nk
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "data").is_dir():
    ROOT = ROOT.parent
DATA_PATH = ROOT / "data" / "processed_dataset.npz"
OUTPUT_PATH = ROOT / "data" / "cleaned_dataset.npz"
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Dataset archive not found: {DATA_PATH}")

sys.path.insert(0, str(ROOT))
from data.bp_utils import extract_bp_labels

FS = 125
WINDOW_SAMPLES = 625
MAX_ZERO_FRACTION = 0.05
MIN_STD = 1e-6
ABP_LOW_PASS_HZ = 40
SBP_RANGE = (60.0, 250.0)
DBP_RANGE = (30.0, 150.0)


## 1. Load aligned source windows

The archive retains exact duplicate recordings for provenance. Windows marked `excluded` are omitted so duplicate copies cannot leak across data splits. PPG, ABP and all window metadata use the same row mask.

In [7]:
required = {"ppg", "abp", "window_ids", "source_record_ids", "split_assignments"}
with np.load(DATA_PATH, allow_pickle=False) as archive:
    missing = required.difference(archive.files)
    if missing:
        raise ValueError(f"Archive is missing required arrays: {sorted(missing)}")
    split_all = archive["split_assignments"]
    keep_source = split_all != "excluded"
    ppg = np.asarray(archive["ppg"][keep_source], dtype=np.float32)
    abp = np.asarray(archive["abp"][keep_source], dtype=np.float32)
    window_ids = archive["window_ids"][keep_source]
    source_record_ids = archive["source_record_ids"][keep_source]
    splits = split_all[keep_source]

if ppg.ndim != 2 or ppg.shape[1] != WINDOW_SAMPLES or abp.shape != ppg.shape:
    raise ValueError(f"Expected aligned (n_windows, {WINDOW_SAMPLES}) PPG/ABP arrays; got {ppg.shape} and {abp.shape}")
if not (len(window_ids) == len(source_record_ids) == len(splits) == len(ppg)):
    raise ValueError("Signal and metadata row counts are not aligned")

print(f"Excluded duplicate-copy windows: {int((~keep_source).sum()):,}")
print(f"Windows to process: {len(ppg):,}")
print(f"PPG/ABP shape: {ppg.shape}")

Excluded duplicate-copy windows: 700
Windows to process: 528,128
PPG/ABP shape: (528128, 625)


## 2. Clean ABP and PPG

For each row, the notebook applies the same initial quality checks to both signals: all samples must be finite, the standard deviation must be at least `MIN_STD`, and no more than 5% of samples may be exactly zero. Small numbers of zeros are linearly interpolated.

ABP is filtered with NeuroKit2's 4th-order Butterworth low-pass at 40 Hz. PPG uses `nk.ppg_process` at 125 Hz; this does cleaning, pulse detection, rate estimation and quality estimation. Processing exceptions and bad outputs are counted and reported. A rejected row is excluded from PPG and ABP together.

In [8]:
def interpolate_zeros(signal):
    """Linearly interpolate zero-valued samples, including edge zeros."""
    zero = signal == 0
    if zero.any() and not zero.all():
        positions = np.arange(signal.size)
        signal = np.interp(positions, positions[~zero], signal[~zero])
    return signal


def raw_signal_is_usable(signal):
    return (
        np.isfinite(signal).all()
        and np.mean(signal == 0) <= MAX_ZERO_FRACTION
        and np.std(signal) >= MIN_STD
    )

n_windows = len(ppg)
row_valid = np.zeros(n_windows, dtype=bool)
sbp = np.full(n_windows, np.nan, dtype=np.float32)
dbp = np.full(n_windows, np.nan, dtype=np.float32)
abp_peak_counts = np.zeros(n_windows, dtype=np.int16)
abp_valley_counts = np.zeros(n_windows, dtype=np.int16)
ppg_peak_counts = np.zeros(n_windows, dtype=np.int16)
dropped_raw = 0
dropped_labels = 0
processing_errors = Counter()

for i in range(n_windows):
    if i % 10_000 == 0:
        print(f"Processing window {i:,} / {n_windows:,} ...")

    if not raw_signal_is_usable(ppg[i]) or not raw_signal_is_usable(abp[i]):
        dropped_raw += 1
        continue

    try:
        ppg_input = interpolate_zeros(ppg[i].astype(np.float64, copy=True))
        abp_input = interpolate_zeros(abp[i].astype(np.float64, copy=True))

        ppg_signals, _ = nk.ppg_process(ppg_input, sampling_rate=FS)
        cleaned_ppg = ppg_signals["PPG_Clean"].to_numpy(dtype=np.float64)
        ppg_peak_counts[i] = int((ppg_signals["PPG_Peaks"].to_numpy() > 0).sum())

        cleaned_abp = np.asarray(
            nk.signal_filter(
                abp_input,
                sampling_rate=FS,
                highcut=ABP_LOW_PASS_HZ,
                method="butterworth",
                order=4,
            ),
            dtype=np.float64,
        )
        if not (np.isfinite(cleaned_ppg).all() and np.isfinite(cleaned_abp).all()):
            raise ValueError("NeuroKit returned non-finite samples")

        sbp_i, dbp_i, n_peaks, n_valleys = extract_bp_labels(cleaned_abp)
    except Exception as error:
        processing_errors[f"{type(error).__name__}: {error}"] += 1
        continue

    abp_peak_counts[i] = n_peaks
    abp_valley_counts[i] = n_valleys
    labels_ok = (
        np.isfinite(sbp_i)
        and np.isfinite(dbp_i)
        and SBP_RANGE[0] <= sbp_i <= SBP_RANGE[1]
        and DBP_RANGE[0] <= dbp_i <= DBP_RANGE[1]
        and sbp_i > dbp_i
    )
    if not labels_ok:
        dropped_labels += 1
        continue

    ppg[i] = cleaned_ppg.astype(np.float32)
    abp[i] = cleaned_abp.astype(np.float32)
    sbp[i] = sbp_i
    dbp[i] = dbp_i
    row_valid[i] = True

print("Cleaning complete")
print(f"Dropped by raw signal checks: {dropped_raw:,}")
print(f"Dropped by SBP/DBP checks: {dropped_labels:,}")
print(f"Dropped by NeuroKit/processing errors: {sum(processing_errors.values()):,}")
if processing_errors:
    for message, count in processing_errors.most_common():
        print(f"  {count:,} x {message}")
print(f"Retained windows: {row_valid.sum():,} / {n_windows:,}")

Processing window 0 / 528,128 ...


c:\Users\ingri\OneDrive - The Ohio State University\Projects\Microsoft x BTT\.venv\Lib\site-packages\neurokit2\signal\signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(


Processing window 10,000 / 528,128 ...
Processing window 20,000 / 528,128 ...
Processing window 30,000 / 528,128 ...
Processing window 40,000 / 528,128 ...


c:\Users\ingri\OneDrive - The Ohio State University\Projects\Microsoft x BTT\.venv\Lib\site-packages\neurokit2\signal\signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(


Processing window 50,000 / 528,128 ...
Processing window 60,000 / 528,128 ...
Processing window 70,000 / 528,128 ...


...

Processing window 470,000 / 528,128 ...
Processing window 480,000 / 528,128 ...
Processing window 490,000 / 528,128 ...


c:\Users\ingri\OneDrive - The Ohio State University\Projects\Microsoft x BTT\.venv\Lib\site-packages\neurokit2\signal\signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
c:\Users\ingri\OneDrive - The Ohio State University\Projects\Microsoft x BTT\.venv\Lib\site-packages\neurokit2\signal\signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(


Processing window 500,000 / 528,128 ...
Processing window 510,000 / 528,128 ...


c:\Users\ingri\OneDrive - The Ohio State University\Projects\Microsoft x BTT\.venv\Lib\site-packages\neurokit2\signal\signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
c:\Users\ingri\OneDrive - The Ohio State University\Projects\Microsoft x BTT\.venv\Lib\site-packages\neurokit2\signal\signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
c:\Users\ingri\OneDrive - The Ohio State University\Projects\Microsoft x BTT\.venv\Lib\site-packages\neurokit2\signal\signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
c:\Users\ingri\OneDrive - The Ohio State University\Projects\Microsoft x BTT\.venv\Lib\site-packages\neurokit2\signal\signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
c:\Users\ingri\OneDrive - The Ohio State University\Projects\Microsoft x

Processing window 520,000 / 528,128 ...
Cleaning complete
Dropped by raw signal checks: 53
Dropped by SBP/DBP checks: 1,491
Dropped by NeuroKit/processing errors: 90
  90 x ValueError: cannot convert float NaN to integer
Retained windows: 526,494 / 528,128


## 3. Keep only rows that passed, preserving alignment

One shared mask filters cleaned PPG, cleaned ABP, labels, IDs and split assignments. Peak counts are retained as quality-control metadata, not model features.

In [9]:
ppg_out = ppg[row_valid].copy()
abp_out = abp[row_valid].copy()
sbp_out = sbp[row_valid]
dbp_out = dbp[row_valid]
ids_out = window_ids[row_valid]
records_out = source_record_ids[row_valid]
splits_out = splits[row_valid]
ppg_peak_counts_out = ppg_peak_counts[row_valid]
abp_peak_counts_out = abp_peak_counts[row_valid]
abp_valley_counts_out = abp_valley_counts[row_valid]

n_out = len(ppg_out)
if not all(len(values) == n_out for values in (
    abp_out, sbp_out, dbp_out, ids_out, records_out, splits_out,
    ppg_peak_counts_out, abp_peak_counts_out, abp_valley_counts_out,
)):
    raise ValueError("Filtered signal, label and metadata rows lost alignment")

print(f"Output PPG/ABP shape: {ppg_out.shape}")
for split in ("train", "validation", "test"):
    print(f"{split}: {(splits_out == split).sum():,} windows")

Output PPG/ABP shape: (526494, 625)
train: 371,212 windows
validation: 76,732 windows
test: 78,550 windows


## 4. Save the cleaned datasets

Writes a separate archive; the original `processed_dataset.npz` is not modified.

In [10]:
np.savez_compressed(
    OUTPUT_PATH,
    ppg=ppg_out,
    abp=abp_out,
    sbp=sbp_out,
    dbp=dbp_out,
    window_ids=ids_out,
    source_record_ids=records_out,
    split_assignments=splits_out,
    ppg_peak_counts=ppg_peak_counts_out,
    abp_peak_counts=abp_peak_counts_out,
    abp_valley_counts=abp_valley_counts_out,
)
print(f"Saved {n_out:,} aligned windows to {OUTPUT_PATH}")

Saved 526,494 aligned windows to c:\Users\ingri\OneDrive - The Ohio State University\Projects\Microsoft x BTT\Microsoft-1D-educational-blood-pressure-estimation\data\cleaned_dataset.npz
